# IT3190 — Early Warning System for Student Dropout

**Task:** multi-class classification: `Dropout` / `Enrolled` / `Graduate`.

**Dataset:** *Predict Students' Dropout and Academic Success* (UCI dataset id 697).  
Kaggle mirror: `mattop/predict-students-dropout-and-academic-success`.

This notebook implements two experiments:

1. **Admission-time**: use only information available at enrollment; remove semester 1 and semester 2 academic features.
2. **Semester-1 early warning**: add first-semester academic features; still remove semester 2 features.

Main metric: **Macro-F1**. We also report Accuracy, Balanced Accuracy, per-class metrics, and especially **Dropout Recall**.

### Run on Kaggle
1. Create a Kaggle Notebook.
2. `Add Input` → search for **Predict Students' Dropout and Academic Success** by `mattop`.
3. Run all cells. The loader auto-detects `dataset.csv`/`data.csv` under the Kaggle input directory.

Local alternative:
```bash
kaggle datasets download -d mattop/predict-students-dropout-and-academic-success -p data --unzip
```


In [ ]:
# Optional local install (uncomment if needed)
# !pip install -q pandas numpy scikit-learn matplotlib joblib ucimlrepo

from pathlib import Path
import os, random, json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score
)
from sklearn.model_selection import (
    RandomizedSearchCV, StratifiedKFold, cross_validate, train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
TARGET_COL = "Target"
CLASS_ORDER = ["Dropout", "Enrolled", "Graduate"]
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)


In [ ]:
def read_csv_robust(path):
    path = Path(path)
    df = pd.read_csv(path)
    if df.shape[1] == 1:  # some mirrors use semicolon delimiter
        df = pd.read_csv(path, sep=";")
    return df


def load_dataset(explicit_path=None):
    candidates = []
    if explicit_path:
        candidates.append(Path(explicit_path))
    if os.getenv("DATASET_PATH"):
        candidates.append(Path(os.environ["DATASET_PATH"]))

    candidates += [
        Path("/kaggle/input/predict-students-dropout-and-academic-success/dataset.csv"),
        Path("/kaggle/input/predict-students-dropout-and-academic-success/data.csv"),
        Path("data/dataset.csv"), Path("data/data.csv"),
        Path("dataset.csv"), Path("data.csv"),
    ]

    path = next((p for p in candidates if p.is_file()), None)
    if path is None:
        kaggle_dir = Path("/kaggle/input/predict-students-dropout-and-academic-success")
        if kaggle_dir.exists():
            csvs = sorted(kaggle_dir.rglob("*.csv"))
            path = csvs[0] if csvs else None

    if path is not None:
        df = read_csv_robust(path)
        source = str(path)
    else:
        try:
            from ucimlrepo import fetch_ucirepo
        except ImportError as e:
            raise FileNotFoundError(
                "Dataset not found. Attach the Kaggle dataset, provide a local CSV, "
                "or install ucimlrepo."
            ) from e
        ds = fetch_ucirepo(id=697)
        X = ds.data.features.copy()
        y = ds.data.targets.copy()
        if isinstance(y, pd.Series):
            y = y.to_frame(name=TARGET_COL)
        elif TARGET_COL not in y.columns and y.shape[1] == 1:
            y.columns = [TARGET_COL]
        df = pd.concat([X.reset_index(drop=True), y.reset_index(drop=True)], axis=1)
        source = "UCI via ucimlrepo (id=697)"

    df.columns = [str(c).strip() for c in df.columns]
    df = df.drop(columns=[c for c in df.columns if c.lower().startswith("unnamed")], errors="ignore")

    if TARGET_COL not in df.columns:
        candidate = [c for c in df.columns if c.lower() == "target"]
        if not candidate:
            raise ValueError("Target column not found")
        df = df.rename(columns={candidate[0]: TARGET_COL})

    target_map = {"dropout": "Dropout", "enrolled": "Enrolled", "graduate": "Graduate"}
    mapped = df[TARGET_COL].astype(str).str.strip().str.lower().map(target_map)
    if mapped.isna().any():
        raise ValueError(f"Unexpected target labels: {df.loc[mapped.isna(), TARGET_COL].unique()}")
    df[TARGET_COL] = mapped
    return df, source


df, DATA_SOURCE = load_dataset()
print("Source:", DATA_SOURCE)
print("Shape:", df.shape)
display(df.head())


In [ ]:
# Basic data audit
print(df.info())
print("\nMissing values:", int(df.isna().sum().sum()))
print("\nDuplicate rows:", int(df.duplicated().sum()))
print("\nTarget counts:")
display(df[TARGET_COL].value_counts().to_frame("count"))

fig, ax = plt.subplots(figsize=(6, 4))
counts = df[TARGET_COL].value_counts().reindex(CLASS_ORDER)
ax.bar(counts.index, counts.values)
ax.set_title("Target class distribution")
ax.set_ylabel("Students")
plt.show()


In [ ]:
CATEGORICAL_COLUMNS = [
    "Marital status", "Application mode", "Course", "Daytime/evening attendance",
    "Previous qualification", "Nacionality", "Nationality",
    "Mother's qualification", "Father's qualification",
    "Mother's occupation", "Father's occupation",
    "Displaced", "Educational special needs", "Debtor",
    "Tuition fees up to date", "Gender", "Scholarship holder", "International",
]

SEM1_PREFIX = "Curricular units 1st sem"
SEM2_PREFIX = "Curricular units 2nd sem"

all_features = [c for c in df.columns if c != TARGET_COL]
FEATURE_SETS = {
    "admission": [c for c in all_features if not c.startswith(SEM1_PREFIX) and not c.startswith(SEM2_PREFIX)],
    "semester1": [c for c in all_features if not c.startswith(SEM2_PREFIX)],
}

for name, cols in FEATURE_SETS.items():
    print(f"{name:10s}: {len(cols)} features")
    print(cols)
    print()


In [ ]:
def make_preprocessor(feature_columns):
    feature_columns = list(feature_columns)
    categorical = [c for c in CATEGORICAL_COLUMNS if c in feature_columns]
    numerical = [c for c in feature_columns if c not in categorical]

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("num", num_pipe, numerical),
        ("cat", cat_pipe, categorical),
    ])


def make_pipeline(feature_columns, estimator):
    return Pipeline([
        ("preprocess", make_preprocessor(feature_columns)),
        ("model", estimator),
    ])


def get_baselines():
    return {
        "Dummy": DummyClassifier(strategy="most_frequent"),
        "LogisticRegression": LogisticRegression(
            max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE
        ),
        "DecisionTree": DecisionTreeClassifier(
            max_depth=10, min_samples_leaf=4,
            class_weight="balanced", random_state=RANDOM_STATE
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300, max_features="sqrt",
            class_weight="balanced_subsample", random_state=RANDOM_STATE,
            n_jobs=-1
        ),
    }


In [ ]:
# Fixed stratified hold-out test set shared by both feature settings.
X_all = df.drop(columns=[TARGET_COL])
y_all = df[TARGET_COL]
indices = np.arange(len(df))

train_idx, test_idx = train_test_split(
    indices, test_size=0.20, stratify=y_all, random_state=RANDOM_STATE
)
X_train, X_test = X_all.iloc[train_idx].copy(), X_all.iloc[test_idx].copy()
y_train, y_test = y_all.iloc[train_idx].copy(), y_all.iloc[test_idx].copy()

print("Train:", X_train.shape, y_train.value_counts().to_dict())
print("Test :", X_test.shape, y_test.value_counts().to_dict())


In [ ]:
def compare_models_cv(feature_setting, feature_columns, cv_splits=5):
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    scoring = {
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "macro_f1": "f1_macro",
    }
    rows = []
    for model_name, estimator in get_baselines().items():
        pipe = make_pipeline(feature_columns, estimator)
        scores = cross_validate(
            pipe, X_train[feature_columns], y_train,
            cv=cv, scoring=scoring, n_jobs=-1
        )
        rows.append({
            "feature_setting": feature_setting,
            "model": model_name,
            "cv_accuracy_mean": scores["test_accuracy"].mean(),
            "cv_accuracy_std": scores["test_accuracy"].std(),
            "cv_balanced_accuracy_mean": scores["test_balanced_accuracy"].mean(),
            "cv_macro_f1_mean": scores["test_macro_f1"].mean(),
            "cv_macro_f1_std": scores["test_macro_f1"].std(),
        })
    return pd.DataFrame(rows).sort_values("cv_macro_f1_mean", ascending=False)

cv_results = pd.concat([
    compare_models_cv("admission", FEATURE_SETS["admission"]),
    compare_models_cv("semester1", FEATURE_SETS["semester1"]),
], ignore_index=True)

display(cv_results.sort_values("cv_macro_f1_mean", ascending=False))


In [ ]:
# Visualize baseline CV Macro-F1
plot_df = cv_results.sort_values("cv_macro_f1_mean", ascending=True).copy()
labels = plot_df["feature_setting"] + " / " + plot_df["model"]

fig, ax = plt.subplots(figsize=(10, 6))
ypos = np.arange(len(plot_df))
ax.barh(ypos, plot_df["cv_macro_f1_mean"], xerr=plot_df["cv_macro_f1_std"])
ax.set_yticks(ypos, labels=labels)
ax.set_xlim(0, 1)
ax.set_xlabel("5-fold CV Macro-F1")
ax.set_title("Model comparison")
plt.show()


In [ ]:
def tune_rf(feature_columns, n_iter=24, cv_splits=5):
    pipe = make_pipeline(
        feature_columns,
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
    )
    param_distributions = {
        "model__n_estimators": [200, 300, 500, 700],
        "model__max_depth": [None, 8, 12, 16, 24],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", "log2", 0.5],
        "model__class_weight": [None, "balanced", "balanced_subsample"],
    }
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    search = RandomizedSearchCV(
        pipe, param_distributions=param_distributions,
        n_iter=n_iter, scoring="f1_macro", refit=True,
        cv=cv, random_state=RANDOM_STATE, n_jobs=-1, verbose=1
    )
    search.fit(X_train[feature_columns], y_train)
    return search

# Tune each feature setting independently using training-only CV.
# Reduce n_iter to ~8 for a quick smoke test.
search_admission = tune_rf(FEATURE_SETS["admission"], n_iter=24)
search_sem1 = tune_rf(FEATURE_SETS["semester1"], n_iter=24)

print("Admission best CV Macro-F1:", search_admission.best_score_)
print("Admission params:", search_admission.best_params_)
print()
print("Semester-1 best CV Macro-F1:", search_sem1.best_score_)
print("Semester-1 params:", search_sem1.best_params_)


In [ ]:
def evaluate(model, X, y):
    pred = model.predict(X)
    metrics = {
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "macro_precision": precision_score(y, pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y, pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
        "dropout_recall": recall_score(y == "Dropout", pred == "Dropout", zero_division=0),
        "dropout_f1": f1_score(y == "Dropout", pred == "Dropout", zero_division=0),
    }
    report = pd.DataFrame(classification_report(
        y, pred, labels=CLASS_ORDER, output_dict=True, zero_division=0
    )).T
    cm = confusion_matrix(y, pred, labels=CLASS_ORDER)
    return metrics, report, cm

experiment_rows = []
for name, search in [("admission", search_admission), ("semester1", search_sem1)]:
    features = FEATURE_SETS[name]
    metrics, _, _ = evaluate(search.best_estimator_, X_test[features], y_test)
    experiment_rows.append({"feature_setting": name, "best_cv_macro_f1": search.best_score_, **metrics})

experiment_results = pd.DataFrame(experiment_rows)
display(experiment_results)


In [ ]:
# Final project model: Semester-1 early-warning Random Forest.
# We define this setting a priori because the project goal is early warning after semester 1.
final_features = FEATURE_SETS["semester1"]
final_model = search_sem1.best_estimator_
final_metrics, final_report, final_cm = evaluate(
    final_model, X_test[final_features], y_test
)

print(json.dumps(final_metrics, indent=2))
display(final_report)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(final_cm)
ax.set_xticks(range(3), labels=CLASS_ORDER, rotation=25, ha="right")
ax.set_yticks(range(3), labels=CLASS_ORDER)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix — Semester-1 RF")
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(final_cm[i, j]), ha="center", va="center")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()


In [ ]:
# Permutation importance on the original raw features.
perm = permutation_importance(
    final_model,
    X_test[final_features], y_test,
    scoring="f1_macro", n_repeats=10,
    random_state=RANDOM_STATE, n_jobs=-1
)
importance_df = pd.DataFrame({
    "feature": final_features,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

display(importance_df.head(20))

top = importance_df.head(20).sort_values("importance_mean", ascending=True)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top["feature"], top["importance_mean"], xerr=top["importance_std"])
ax.set_xlabel("Decrease in Macro-F1 after permutation")
ax.set_title("Top-20 feature importance")
plt.show()


In [ ]:
# Save reproducible artifacts
OUTPUT_DIR = Path("artifacts")
OUTPUT_DIR.mkdir(exist_ok=True)

cv_results.to_csv(OUTPUT_DIR / "model_comparison_cv.csv", index=False)
experiment_results.to_csv(OUTPUT_DIR / "feature_setting_test_results.csv", index=False)
final_report.to_csv(OUTPUT_DIR / "classification_report_semester1.csv")
importance_df.to_csv(OUTPUT_DIR / "permutation_importance_semester1.csv", index=False)
joblib.dump(final_model, OUTPUT_DIR / "student_dropout_semester1_rf.joblib")

metadata = {
    "dataset_source": DATA_SOURCE,
    "dataset_shape": list(df.shape),
    "target_distribution": df[TARGET_COL].value_counts().to_dict(),
    "random_state": RANDOM_STATE,
    "feature_sets": FEATURE_SETS,
    "final_feature_setting": "semester1",
    "best_cv_macro_f1": float(search_sem1.best_score_),
    "best_params": search_sem1.best_params_,
    "test_metrics": {k: float(v) for k, v in final_metrics.items()},
}
with open(OUTPUT_DIR / "metrics_and_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Saved to:", OUTPUT_DIR.resolve())


## What to report in the IT3190 presentation

Use the notebook outputs to build these tables/figures:

- Target class distribution.
- 5-fold CV comparison: Dummy vs Logistic Regression vs Decision Tree vs Random Forest.
- **Ablation:** Admission-only vs Semester-1 early warning.
- Hold-out metrics: Accuracy, Balanced Accuracy, Macro-F1, Dropout Recall.
- Confusion matrix.
- Permutation feature importance.

**Important interpretation:** feature importance is predictive association, not causal evidence. A feature being useful for prediction does not mean it causes dropout.
